# Q2

## Q2-1

In [1]:
# import
import pyspark
from pyspark.sql import SparkSession, SQLContext
from pyspark.ml import Pipeline, Transformer
from pyspark.ml.feature import Imputer, StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.sql.functions import *
from pyspark.sql.types import *
import numpy as np

# column config
col_names = ["duration","protocol_type","service","flag","src_bytes",
"dst_bytes","land","wrong_fragment","urgent","hot","num_failed_logins",
"logged_in","num_compromised","root_shell","su_attempted","num_root",
"num_file_creations","num_shells","num_access_files","num_outbound_cmds",
"is_host_login","is_guest_login","count","srv_count","serror_rate",
"srv_serror_rate","rerror_rate","srv_rerror_rate","same_srv_rate",
"diff_srv_rate","srv_diff_host_rate","dst_host_count","dst_host_srv_count",
"dst_host_same_srv_rate","dst_host_diff_srv_rate","dst_host_same_src_port_rate",
"dst_host_srv_diff_host_rate","dst_host_serror_rate","dst_host_srv_serror_rate",
"dst_host_rerror_rate","dst_host_srv_rerror_rate","class","difficulty"]

nominal_cols = ['protocol_type','service','flag']
binary_cols = ['land', 'logged_in', 'root_shell', 'su_attempted', 'is_host_login','is_guest_login']
continuous_cols = ['duration' ,'src_bytes', 'dst_bytes', 'wrong_fragment','urgent', 'hot',
'num_failed_logins', 'num_compromised', 'num_root' ,'num_file_creations',
'num_shells', 'num_access_files', 'num_outbound_cmds', 'count' ,'srv_count',
'serror_rate', 'srv_serror_rate' ,'rerror_rate' ,'srv_rerror_rate',
'same_srv_rate', 'diff_srv_rate', 'srv_diff_host_rate' ,'dst_host_count',
'dst_host_srv_count' ,'dst_host_same_srv_rate' ,'dst_host_diff_srv_rate',
'dst_host_same_src_port_rate' ,'dst_host_srv_diff_host_rate',
'dst_host_serror_rate' ,'dst_host_srv_serror_rate', 'dst_host_rerror_rate',
'dst_host_srv_rerror_rate']

# transformer
class OutcomeCreater(Transformer):
    def __init__(self):
        super().__init__()
    def _transform(self, dataset):
        label_to_binary = udf(lambda name: 0.0 if name == 'normal' else 1.0)
        output_df = dataset.withColumn('outcome', label_to_binary(col('class'))).drop("class")
        output_df = output_df.withColumn('outcome', col('outcome').cast(DoubleType()))
        output_df = output_df.drop('difficulty')
        return output_df

class FeatureTypeCaster(Transformer):
    def __init__(self):
        super().__init__()
    def _transform(self, dataset):
        output_df = dataset
        for c in binary_cols + continuous_cols:
            output_df = output_df.withColumn(c, col(c).cast(DoubleType()))
        return output_df

class ColumnDropper(Transformer):
    def __init__(self, columns_to_drop=None):
        super().__init__()
        self.columns_to_drop = columns_to_drop or []
    def _transform(self, dataset):
        output_df = dataset
        for c in self.columns_to_drop:
            output_df = output_df.drop(c)
        return output_df
        from pyspark.ml.feature import StringIndexer

spark = SparkSession.builder.master("local[*]").appName("Q2").getOrCreate()
train_raw = spark.read.csv('KDDTrain+.txt', header=False).toDF(*col_names)
test_raw = spark.read.csv('KDDTest+.txt',  header=False).toDF(*col_names)

DOS=['back','land','neptune','pod','smurf','teardrop','apache2','udpstorm','processtable','mailbomb']
probing=['ipsweep','nmap','portsweep','satan','mscan','saint']
U2R=['buffer_overflow','loadmodule','perl','rootkit','ps','sqlattack','xterm']

def add_attack(df):
    c = lower(col("class"))
    return (df.withColumn(
              "attack",
              when(c=='normal','normal')
              .when(c.isin(DOS),   'DOS')
              .when(c.isin(probing), 'probing')
              .when(c.isin(U2R),   'U2R')
              .otherwise('R2L'))
           )

train_raw2 = add_attack(train_raw)
test_raw2 = add_attack(test_raw)
label_indexer = StringIndexer(inputCol="attack", outputCol="label", handleInvalid="keep")
label_model = label_indexer.fit(train_raw2)
train_idx = label_model.transform(train_raw2)
test_idx = label_model.transform(test_raw2)
def get_preprocess_pipeline():
    stage_typecaster = FeatureTypeCaster()
    nominal_id_cols = [x + "_index" for x in nominal_cols]
    nominal_oh_cols = [x + "_encoded" for x in nominal_cols]
    stage_nominal_indexer = StringIndexer(inputCols=nominal_cols, outputCols=nominal_id_cols, handleInvalid="keep")
    stage_nominal_onehot = OneHotEncoder(inputCols=nominal_id_cols, outputCols=nominal_oh_cols, handleInvalid="keep")
    feature_cols = continuous_cols + binary_cols + nominal_oh_cols
    corelated_cols_to_remove = ["dst_host_serror_rate","srv_serror_rate","dst_host_srv_serror_rate",
                                "srv_rerror_rate","dst_host_rerror_rate","dst_host_srv_rerror_rate"]
    feature_cols= [c for c in feature_cols if c not in corelated_cols_to_remove]
    stage_vec= VectorAssembler(inputCols=feature_cols, outputCol="vectorized_features", handleInvalid="keep")
    stage_scaler= StandardScaler(inputCol="vectorized_features", outputCol="features")
    stage_drop= ColumnDropper(columns_to_drop = nominal_cols + nominal_id_cols + nominal_oh_cols +
                                 binary_cols + continuous_cols + ['vectorized_features','difficulty'])
    return Pipeline(stages=[stage_typecaster, stage_nominal_indexer, stage_nominal_onehot,
                            stage_vec, stage_scaler, stage_drop])
feat_pipe = get_preprocess_pipeline()
feat_model = feat_pipe.fit(train_idx)
train_df = feat_model.transform(train_idx)
test_df = feat_model.transform(test_idx)

train_df.printSchema()
train_df.select("label","attack").show(20, truncate=False)

root
 |-- class: string (nullable = true)
 |-- attack: string (nullable = false)
 |-- label: double (nullable = false)
 |-- features: vector (nullable = true)

+-----+-------+
|label|attack |
+-----+-------+
|0.0  |normal |
|0.0  |normal |
|1.0  |DOS    |
|0.0  |normal |
|0.0  |normal |
|1.0  |DOS    |
|1.0  |DOS    |
|1.0  |DOS    |
|1.0  |DOS    |
|1.0  |DOS    |
|1.0  |DOS    |
|1.0  |DOS    |
|0.0  |normal |
|3.0  |R2L    |
|1.0  |DOS    |
|1.0  |DOS    |
|0.0  |normal |
|2.0  |probing|
|0.0  |normal |
|0.0  |normal |
+-----+-------+
only showing top 20 rows



## Q2-2

In [2]:
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.feature import IndexToString, StringIndexerModel
from pyspark.sql import functions as F

evaluator_acc = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="accuracy"
)

# Logistic Regression
predictions_train_lr = lr_model.transform(train_df)
predictions_test_lr = lr_model.transform(test_df)

# Train accuracy
accuracy_train = (
    predictions_train_lr.filter(F.col("label") == F.col("prediction"))
    .count()
    / float(predictions_train_lr.count())
)

# Test accuracy
accuracy_test = (
    predictions_test_lr.filter(F.col("label") == F.col("prediction"))
    .count()
    / float(predictions_test_lr.count())
)

print(f"[Random Forest] Train Accuracy : {np.round(accuracy_train * 100, 2)}%")
print(f"[Random Forest] Test Accuracy  : {np.round(accuracy_test * 100, 2)}%")

NameError: name 'lr_model' is not defined

In [ ]:
predictions_train_rf = rf_model.transform(train_df)
predictions_test_rf  = rf_model.transform(test_df)

accuracy_train_rf = (predictions_train_rf.filter(F.col("label")==F.col("prediction")).count()
                / float(predictions_train_rf.count()))
accuracy_test_rf  = (predictions_test_rf.filter(F.col("label")==F.col("prediction")).count()
                / float(predictions_test_rf.count()))

print(f"[Logistic Regression] Train Accuracy : {np.round(accuracy_train_rf*100,2)}%")
print(f"[Logistic Regression] Test  Accuracy : {np.round(accuracy_test_rf*100,2)}%")

In [ ]:
def confusion_matrix_named(pred_df: DataFrame, labels: list, true_col="attack"):
    to_pred = IndexToString(inputCol="prediction", outputCol="pred_attack", labels=labels)
    named = to_pred.transform(pred_df)
    return (named.groupBy(true_col, "pred_attack").count()
                 .groupBy(true_col)
                 .pivot("pred_attack", labels)
                 .sum("count")
                 .na.fill(0)
                 .orderBy(true_col))

labels_list = label_model.labels

cm_lr = confusion_matrix_named(lr_pred_test, labels_list)
cm_rf = confusion_matrix_named(rf_pred_test, labels_list)

print("LR Confusion Matrix (attack names):")
cm_lr.show(truncate=False)
print("RF Confusion Matrix (attack names):")
cm_rf.show(truncate=False)